In [1]:
import glob
import os
import shutil
import tempfile
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn.functional as F
from monai.config import print_config
from monai.data import DataLoader, Dataset, CacheDataset
from monai.data.image_reader import NibabelReader, ITKReader
from monai.inferers import sliding_window_inference
from monai.losses import PatchAdversarialLoss, PerceptualLoss
from monai.networks.nets import AutoencoderKL, DiffusionModelUNet, PatchDiscriminator
from monai.networks.schedulers import DDPMScheduler
from monai.transforms import (
    Compose,
    EnsureChannelFirstd,
    LoadImaged,
    Orientationd,
    RandSpatialCropd,
    ScaleIntensityRanged,
    Spacingd,
    ToTensord,
)
from monai.utils import set_determinism
from sklearn.model_selection import train_test_split
from tqdm import tqdm

print_config()

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

2026-01-06 16:43:28.262879: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: SSE3 SSE4.1 SSE4.2 AVX AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-01-06 16:43:29.342206: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.


MONAI version: 1.5.1
Numpy version: 2.3.5
Pytorch version: 2.9.1
MONAI flags: HAS_EXT = False, USE_COMPILED = False, USE_META_DICT = False
MONAI rev id: 9c6d819f97e37f36c72f3bdfad676b455bd2fa0d
MONAI __file__: /home/<username>/projects/ood-detection/.venv/lib/python3.13/site-packages/monai/__init__.py

Optional dependencies:
Pytorch Ignite version: NOT INSTALLED or UNKNOWN VERSION.
ITK version: 5.4.4
Nibabel version: 5.3.2
scikit-image version: 0.25.2
scipy version: 1.16.3
Pillow version: 12.0.0
Tensorboard version: 2.20.0
gdown version: NOT INSTALLED or UNKNOWN VERSION.
TorchVision version: 0.24.1
tqdm version: 4.67.1
lmdb version: NOT INSTALLED or UNKNOWN VERSION.
psutil version: 7.1.3
pandas version: 2.3.1
einops version: 0.8.1
transformers version: NOT INSTALLED or UNKNOWN VERSION.
mlflow version: NOT INSTALLED or UNKNOWN VERSION.
pynrrd version: NOT INSTALLED or UNKNOWN VERSION.
clearml version: NOT INSTALLED or UNKNOWN VERSION.

For details about installing the optional dependenc

In [7]:
# Significantly reduced capacity to fit in 4GB VRAM
autoencoder = AutoencoderKL(
    spatial_dims=3,
    in_channels=1,
    out_channels=1,
    channels=(32, 64, 64),
    latent_channels=3,
    num_res_blocks=1,
    attention_levels=(False, False, False),
    with_encoder_nonlocal_attn=False,
    with_decoder_nonlocal_attn=False,
).to(device)

In [8]:
from dataloading.load_atlas import load_atlas

train_loader, val_loader, test_loader = load_atlas()

Found 655 image files
Found 655 mask files
Found 300 test image files
Train data: 524 files
Validation data: 131 files


In [ ]:
# --- Stage 1: Train AutoencoderKL ---
# Note: In a real scenario, train for significantly more epochs

optimizer_ae = torch.optim.Adam(autoencoder.parameters(), lr=1e-4)
recon_loss_fn = torch.nn.MSELoss()
kl_weight = 1e-6

n_epochs_ae = 5  # Reduced for demonstration

print("Starting AutoencoderKL Training...")
autoencoder.train()

for epoch in range(n_epochs_ae):
    epoch_loss = 0
    progress_bar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{n_epochs_ae}")
    for batch in progress_bar:
        images = batch["image"].to(device)
        optimizer_ae.zero_grad()
        
        reconstruction, z_mu, z_sigma = autoencoder(images)
        
        recons_loss = recon_loss_fn(reconstruction, images)
        kl_loss = 0.5 * torch.sum(z_mu.pow(2) + z_sigma.pow(2) - torch.log(z_sigma.pow(2)) - 1, dim=[1, 2, 3, 4])
        kl_loss = torch.mean(kl_loss)
        
        loss = recons_loss + kl_weight * kl_loss
        loss.backward()
        optimizer_ae.step()
        
        epoch_loss += loss.item()
        progress_bar.set_postfix({"loss": loss.item()})
        
    print(f"Epoch {epoch+1} finished. Avg Loss: {epoch_loss / len(train_loader):.4f}")

# Save AE weights
torch.save(autoencoder.state_dict(), "autoencoder.pth")

Starting AutoencoderKL Training...


Epoch 1/5: 100%|██████████| 524/524 [02:01<00:00,  4.32it/s, loss=0.000882]


Epoch 1 finished. Avg Loss: 0.0037


Epoch 2/5: 100%|██████████| 524/524 [02:04<00:00,  4.21it/s, loss=0.000444]


Epoch 2 finished. Avg Loss: 0.0007


...

Epoch 3 finished. Avg Loss: 0.0005


Epoch 4/5: 100%|██████████| 524/524 [02:08<00:00,  4.06it/s, loss=0.00055] 


Epoch 4 finished. Avg Loss: 0.0005


Epoch 5/5: 100%|██████████| 524/524 [02:08<00:00,  4.07it/s, loss=0.00046] 

Epoch 5 finished. Avg Loss: 0.0005


In [9]:
autoencoder.load_state_dict(torch.load("autoencoder.pth"))

<All keys matched successfully>

In [10]:
# Significantly reduced capacity
unet = DiffusionModelUNet(
    spatial_dims=3,
    in_channels=3,
    out_channels=3,
    channels=(32, 64, 128),
    num_res_blocks=1,
    attention_levels=(False, False, True),
).to(device)

scheduler = DDPMScheduler(num_train_timesteps=1000, schedule="linear_beta", beta_start=0.0015, beta_end=0.0195)

In [5]:
# --- Stage 2: Train Diffusion Model ---

optimizer_unet = torch.optim.Adam(unet.parameters(), lr=1e-4)
n_epochs_diff = 10 # Increased epochs
accumulation_steps = 4 # Gradient accumulation for stability

print("Starting Diffusion Model Training...")
autoencoder.eval()
unet.train()

for epoch in range(n_epochs_diff):
    epoch_loss = 0
    progress_bar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{n_epochs_diff}")
    
    optimizer_unet.zero_grad()
    
    for step, batch in enumerate(progress_bar):
        images = batch["image"].to(device)
        
        # 1. Get Latents
        with torch.no_grad():
            z_mu, z_sigma = autoencoder.encode(images)
            z = autoencoder.sampling(z_mu, z_sigma)
            
        # 2. Add Noise
        noise = torch.randn_like(z).to(device)
        timesteps = torch.randint(0, scheduler.num_train_timesteps, (z.shape[0],), device=device).long()
        noisy_z = scheduler.add_noise(original_samples=z, noise=noise, timesteps=timesteps)
        
        # 3. Denoise
        noise_pred = unet(x=noisy_z, timesteps=timesteps)
        
        loss = F.mse_loss(noise_pred, noise)
        
        # Gradient Accumulation
        loss = loss / accumulation_steps
        loss.backward()
        
        if (step + 1) % accumulation_steps == 0:
            torch.nn.utils.clip_grad_norm_(unet.parameters(), 1.0)
            optimizer_unet.step()
            optimizer_unet.zero_grad()
        
        epoch_loss += loss.item() * accumulation_steps
        progress_bar.set_postfix({"loss": loss.item() * accumulation_steps})

    print(f"Epoch {epoch+1} finished. Avg Loss: {epoch_loss / len(train_loader):.4f}")

# Save Diffusion weights
torch.save(unet.state_dict(), "diffusion_unet.pth")

NameError: name 'unet' is not defined

In [11]:
unet.load_state_dict(torch.load("diffusion_unet.pth"))

<All keys matched successfully>

In [12]:
from monai.networks.schedulers import PNDMScheduler

def calculate_confidence_score(image, scheduler):
    with torch.no_grad():
        z_mu, z_sigma = autoencoder.encode(image)
        z = autoencoder.sampling(z_mu, z_sigma)

    mse_scores = []
    mse_loss = torch.nn.MSELoss()

    x_0 = image

    for t in scheduler.timesteps:
        with torch.no_grad():
            # Predict noise residual using the U-Net
            # PNDM expects the time 't' to be passed as a tensor
            model_output = unet(z, timesteps=torch.Tensor((t,)).to(device))
            
            # Compute the previous noisy sample (x_t -> x_t-1)
            # The scheduler handles the complex PLMS math internally
            z, _ = scheduler.step(model_output, t, z)
            x = autoencoder.decode(z)
            mse = mse_loss(x_0, x)
            mse_scores.append(mse)
    confidence_score = sum(mse_scores) / len(mse_scores)
    return confidence_score


In [13]:
import torch.nn.functional as F

def calculate_scores(data_loader):
    scheduler = PNDMScheduler(
        num_train_timesteps=1000, 
        beta_start=0.0015, 
        beta_end=0.0205, 
        prediction_type="epsilon"
    )
    scheduler.set_timesteps(num_inference_steps=50)
    scores = []
    for batch in tqdm(data_loader, desc="Calculating Scores"):
        image = batch["image"].to(device)
        image = F.interpolate(image, size=(64, 64, 64), mode='trilinear', align_corners=False)
        score = calculate_confidence_score(image, scheduler)
        scores.append(score.item())
    return scores

val_scores = calculate_scores(val_loader)

Calculating Scores: 100%|██████████| 131/131 [07:13<00:00,  3.31s/it]


In [14]:
threshold = np.percentile(val_scores, 95)

In [15]:
test_scores = calculate_scores(test_loader)
test_scores[10:]

Calculating Scores: 100%|██████████| 300/300 [16:32<00:00,  3.31s/it]


[0.0015046570915728807,
 0.0018643109360709786,
 0.0021314891055226326,
 0.002479425398632884,
 0.0015539529267698526,
 0.0021881049033254385,
 0.0026181538123637438,
 0.0019877897575497627,
 0.0019487144891172647,
 0.00206926092505455,
 0.0018291270826011896,
 0.0018283640965819359,
 0.0024935323745012283,
 0.0014243453042581677,
 0.0012666378170251846,
 0.0017667789943516254,
 0.0018587691010907292,
 0.0017410790314897895,
 0.00148681306745857,
 0.0017760813934728503,
 0.002252139849588275,
 0.0007546907872892916,
 0.0014944432768970728,
 0.0020889758598059416,
 0.0020169697236269712,
 0.0018628325778990984,
 0.0017568484181538224,
 0.0011816465994343162,
 0.0014824126847088337,
 0.001184686552733183,
 0.002027178183197975,
 0.002184229902923107,
 0.0021506028715521097,
 0.001243945094756782,
 0.0011153174564242363,
 0.0015204580267891288,
 0.0014023398980498314,
 0.0012599688488990068,
 0.0018428791081532836,
 0.001785495551303029,
 0.0015992465196177363,
 0.00228815502487123,
 0.00

In [16]:
from dataloading.load_chaos import load_chaos

chaos_loader = load_chaos()

Found 20 Train CT volumes
Found 20 Test CT volumes
Found 0 Train MR volumes
Found 0 Test MR volumes
Total CHAOS volumes: 40
CHAOS dataloader created with 40 samples


monai.transforms.spatial.dictionary Orientationd.__init__:labels: Current default value of argument `labels=(('L', 'R'), ('P', 'A'), ('I', 'S'))` was changed in version None from `labels=(('L', 'R'), ('P', 'A'), ('I', 'S'))` to `labels=None`. Default value changed to None meaning that the transform now uses the 'space' of a meta-tensor, if applicable, to determine appropriate axis labels.


In [17]:
chaos_scores = calculate_scores(chaos_loader)
print(chaos_scores[:10])

Calculating Scores: 100%|██████████| 40/40 [02:29<00:00,  3.74s/it]

[0.13128341734409332, 0.14269088208675385, 0.12336128950119019, 0.11537524312734604, 0.12293440103530884, 0.11314087361097336, 0.12427212297916412, 0.11980991810560226, 0.12143521755933762, 0.11629986017942429]


In [26]:
scores = np.concatenate([np.array(chaos_scores), np.array(test_scores)])
labels = np.concatenate([np.ones_like(chaos_scores), np.zeros_like(test_scores)])

In [27]:
from sklearn.metrics import average_precision_score
from metrics import fpr

anomaly_scores = scores

aupr = average_precision_score(labels, anomaly_scores)
fpr90 = fpr(labels, anomaly_scores, percentile=10)
fpr95 = fpr(labels, anomaly_scores, percentile=5)
fpr99 = fpr(labels, anomaly_scores, percentile=1)

print(f"AUPR: {aupr:.4f}, fpr90: {fpr90:.4f}, fpr95: {fpr95:.4f}, fpr99: {fpr99:.4f}")

AUPR: 1.0000, fpr90: 0.0000, fpr95: 0.0000, fpr99: 0.0000


In [20]:
from dataloading.load_brats import load_brats
brats_loader = load_brats()

Found 585 Train BRATS volumes
Found 87 Test BRATS volumes


monai.transforms.spatial.dictionary Orientationd.__init__:labels: Current default value of argument `labels=(('L', 'R'), ('P', 'A'), ('I', 'S'))` was changed in version None from `labels=(('L', 'R'), ('P', 'A'), ('I', 'S'))` to `labels=None`. Default value changed to None meaning that the transform now uses the 'space' of a meta-tensor, if applicable, to determine appropriate axis labels.


In [21]:
brats_scores = calculate_scores(brats_loader)
print(brats_scores[:10])

Calculating Scores:   0%|          | 0/672 [00:00<?, ?it/s]WARNING: In /work/ITK-source/ITK/Modules/IO/ImageBase/include/itkImageSeriesReader.hxx, line 478
ImageSeriesReader (0x561b2d954250): Non uniform sampling or missing slices detected,  maximum nonuniformity:0.00101728

ImageSeriesReader (0x561b2d954250): Non uniform sampling or missing slices detected,  maximum nonuniformity:0.000545123

Calculating Scores:   0%|          | 1/672 [00:07<1:22:40,  7.39s/it]WARNING: In /work/ITK-source/ITK/Modules/IO/ImageBase/include/itkImageSeriesReader.hxx, line 478
ImageSeriesReader (0x561b2d954250): Non uniform sampling or missing slices detected,  maximum nonuniformity:0.00100171

Calculating Scores:   3%|▎         | 22/672 [01:16<35:43,  3.30s/it] WARNING: In /work/ITK-source/ITK/Modules/IO/ImageBase/include/itkImageSeriesReader.hxx, line 478
ImageSeriesReader (0x561b2d954250): Non uniform sampling or missing slices detected,  maximum nonuniformity:0.000545999

Calculating Scores:   3%|▎    

[0.31244125962257385, 0.3041066825389862, 0.32286590337753296, 0.33115190267562866, 0.3366318345069885, 0.3599703311920166, 0.24312251806259155, 0.29844680428504944, 0.32934677600860596, 0.3074040710926056]


In [22]:
scores = np.concatenate([np.array(brats_scores), np.array(test_scores)])
labels = np.concatenate([np.ones_like(brats_scores), np.zeros_like(test_scores)])

In [23]:
from sklearn.metrics import average_precision_score
from metrics import fpr

anomaly_scores = scores

aupr = average_precision_score(labels, anomaly_scores)
fpr90 = fpr(labels, anomaly_scores, percentile=10)
fpr95 = fpr(labels, anomaly_scores, percentile=5)
fpr99 = fpr(labels, anomaly_scores, percentile=1)

print(f"AUPR: {aupr:.4f}, fpr90: {fpr90:.4f}, fpr95: {fpr95:.4f}, fpr99: {fpr99:.4f}")

AUPR: 1.0000, fpr90: 0.0000, fpr95: 0.0000, fpr99: 0.0000


In [25]:
from sklearn.metrics import accuracy_score

threshold = np.percentile(val_scores, 95)
accuracy_score(labels, scores > threshold)

0.9804526748971193